In [ ]:
using DelimitedFiles
using Statistics
using Random
using Printf

const OUTPUT_DIR        = "output"

const BURN_IN_LOGLIK    = 0
const THIN_LOGLIK       = 1

function read_csv_matrix(path::String)
    data, hdr = DelimitedFiles.readdlm(path, ',', header=true)
    M = Matrix{Float64}(data)
    header = hdr === nothing ? nothing : vec(collect(hdr))
    return M, header
end

function detect_sim_ids_by_k(outdir::String)
    files = filter(isfile, readdir(outdir; join=true))
    sims = Int[]
    for f in files
        m = match(r"loglik_sim_(\d+)_chain_(\d+)_k(\d+)\.csv", f)
        if m !== nothing
            push!(sims, parse(Int, m.captures[1]))
        end
    end
    return unique(sort(sims))
end

function k_values_for_sim(sim_id::Int, outdir::String)
    files = filter(isfile, readdir(outdir; join=true))
    ks = Int[]
    for f in files
        m = match(Regex("loglik_sim_$(sim_id)_chain_(\\d+)_k(\\d+)\\.csv"), f)
        if m !== nothing
            push!(ks, parse(Int, m.captures[2]))  
        end
    end
    return unique(sort(ks))
end

function load_loglik_for_sim_k(sim_id::Int, k::Int, outdir::String)
    path = joinpath(outdir, "loglik_sim_$(sim_id)_chain_1_k$(k).csv")
    if !isfile(path)
        @warn "Missing loglik file: $path"
        return Array{Float64}(undef, 0, 0)
    end
    M, _ = read_csv_matrix(path)
    if BURN_IN_LOGLIK > 0 || THIN_LOGLIK > 1
        idxs = collect(BURN_IN_LOGLIK+1:THIN_LOGLIK:size(M,1))
        M = M[idxs, :]
    end
    return M
end

function logmeanexp_columnwise(L::AbstractMatrix{<:Real})
    S, T = size(L)
    out = Vector{Float64}(undef, T)
    for j in 1:T
        col = L[:, j]
        m = maximum(col)
        out[j] = m + log(sum(exp.(col .- m)) / S)
    end
    return out
end

function compute_waic_from_matrix(L::AbstractMatrix{<:Real})
    if isempty(L)
        return NaN
    end
    if any(isinf, L)
        @warn "Encountered infinite log-likelihoods. WAIC will be -Inf."
        return -Inf
    end
    lppd = sum(logmeanexp_columnwise(L))
    pwaic = sum(var(L, dims=1))
    return -2 * lppd + 2 * pwaic
end

function write_waic_k_csv(path::String, rows::Vector{Tuple{Int,Int,Float64}})
    # rows: (sim, k, waic)
    open(path, "w") do io
        println(io, "Dataset,kmax,WAIC")
        for (sim, k, w) in rows
            println(io, "$sim,$k,$w")
        end
    end
end

function write_best3_wide_csv(path::String, best_dict::Dict{Int, Vector{Tuple{Int,Float64}}})
    open(path, "w") do io
        println(io, "Dataset,kmin1,WAIC1,kmin2,WAIC2,kmin3,WAIC3")
        for (sim, vec) in sort(collect(best_dict); by=x->x[1])
            k1, w1 = vec[1]
            k2, w2 = length(vec) >= 2 ? vec[2] : (NaN, NaN)
            k3, w3 = length(vec) >= 3 ? vec[3] : (NaN, NaN)
            println(io, "$sim,$k1,$w1,$k2,$w2,$k3,$w3")
        end
    end
end

Random.seed!(2025)

sim_ids = detect_sim_ids_by_k(OUTPUT_DIR)
isempty(sim_ids) && error("No loglik_sim_*_chain_1_k*.csv found in $(OUTPUT_DIR)")

all_waic_rows = Tuple{Int,Int,Float64}[]  # (sim, k, waic)
best3_dict    = Dict{Int, Vector{Tuple{Int,Float64}}}()  # sim => [(k, waic), ...]

for sim in sim_ids
    ks = k_values_for_sim(sim, OUTPUT_DIR)
    isempty(ks) && (@warn "No k files for dataset $sim"; continue)

    waics_this = Tuple{Int,Float64}[]  # (k, waic) for this sim
    @info "Processing dataset $sim ... (k = $(first(ks))..$(last(ks)))"

    for k in ks
        L = load_loglik_for_sim_k(sim, k, OUTPUT_DIR)
        if size(L,1) == 0
            @warn "Empty loglik for dataset $sim, k=$k"
            push!(waics_this, (k, NaN))
            push!(all_waic_rows, (sim, k, NaN))
            continue
        end
        w = compute_waic_from_matrix(L)
        push!(waics_this, (k, w))
        push!(all_waic_rows, (sim, k, w))
    end

    valid = filter(x -> !isnan(x[2]), waics_this)
    if isempty(valid)
        @warn "All WAIC are NaN for dataset $sim"
        continue
    end

    sorted = sort(valid; by = x -> x[2])
    best3_dict[sim] = sorted[1:min(3, length(sorted))]
end

write_waic_k_csv(joinpath(OUTPUT_DIR, "waic_by_dataset_k.csv"), all_waic_rows)
write_best3_wide_csv(joinpath(OUTPUT_DIR, "best3_by_dataset.csv"), best3_dict)

@info "Done. Outputs written in $(OUTPUT_DIR): waic_by_dataset_k.csv, best3_by_dataset.csv"


[ Info: Processing dataset 1 ... (k = 1..30)
[ Info: Processing dataset 2 ... (k = 1..30)
[ Info: Processing dataset 3 ... (k = 1..30)
[ Info: Processing dataset 4 ... (k = 1..30)
[ Info: Processing dataset 5 ... (k = 1..30)
[ Info: Processing dataset 6 ... (k = 1..30)
[ Info: Processing dataset 7 ... (k = 1..30)
[ Info: Processing dataset 8 ... (k = 1..30)
[ Info: Processing dataset 9 ... (k = 1..30)
[ Info: Processing dataset 10 ... (k = 1..30)
[ Info: Processing dataset 11 ... (k = 1..30)
[ Info: Processing dataset 12 ... (k = 1..30)
[ Info: Processing dataset 13 ... (k = 1..30)
[ Info: Processing dataset 14 ... (k = 1..30)
[ Info: Processing dataset 15 ... (k = 1..30)
[ Info: Processing dataset 16 ... (k = 1..30)
[ Info: Processing dataset 17 ... (k = 1..30)
[ Info: Processing dataset 18 ... (k = 1..30)
[ Info: Processing dataset 19 ... (k = 1..30)
[ Info: Processing dataset 20 ... (k = 1..30)
[ Info: Done. Outputs written in output: waic_by_dataset_k.csv, best3_by_dataset.csv
